In [ ]:
#| hide
from fossick import *
import logging
logging.getLogger("scrapling").setLevel(logging.WARNING)

# fossick

> Web search, page fetching, crawling and browser automation for Python programs and AI agents

`fossick` gets text from the web and puts it into Python programs. You can give it a search query, a URL, an arXiv id, a YouTube link, a GitHub repository, or a local file. It returns markdown that a person or a language model can read. AI agents can use it to look up information. Its functions return plain data such as strings, lists, and dictionaries. You can print this data or pass it to a model.

[<code>search</code>](https://vedicreader.github.io/fossick/search.html#search) sends a query to several search engines at once. It merges their answers into one ranked list. `read` takes one target and returns its text. [<code>fetch</code>](https://vedicreader.github.io/fossick/core.html#fetch) downloads a page. `to_md` converts its HTML to markdown. Some sites refuse downloads from scripts. [<code>fetch</code>](https://vedicreader.github.io/fossick/core.html#fetch) can also load a page in a headless browser, a stealth browser, or a copy of Chrome where you are logged in. `cdp_connect` drives that Chrome for pages that need clicks or typing. You can also use these functions from the shell with the `fossick` command. Agents can call them through a Model Context Protocol (MCP) server.

## Install

```sh
uv add fossick
```

Search and plain downloads need nothing else. JavaScript rendering, the stealth fetcher, and `google` use a headless browser through [scrapling](https://github.com/D4Vinci/Scrapling). If these calls raise `BrowserUnavailable`, run `scrapling install` to download its browsers. The browser automation in `fossick.cdp` drives your installed Google Chrome.

## Quick start

These three calls cover most uses. [<code>search</code>](https://vedicreader.github.io/fossick/search.html#search) returns a list of results. Each result is a dictionary with the page `title`, its URL in `href`, and a snippet in `body`. `score` is the merged ranking score. `engines` lists the search engines that returned the page. `method='flashrank'` reorders results with a small relevance model. This method needs `pip install flashrank`. Without `method`, `search` keeps the merged order.

In [ ]:
#| eval: false
results = search('koshas github fts semantic code graph', method='flashrank', n=5)
for r in results: print(r['score'], r['engines'], r['title'], r['href'])

0.030536 ['brave', 'yahoo'] github.com › codegraph-ai › CodeGraphGitHub - codegraph-ai/CodeGraph: CodeGraph builds a semantic... https://github.com/codegraph-ai/CodeGraph
0.016129 ['yahoo'] vedicreader.github.io › kosha › graphgraph – koshas - vedicreader.github.io https://vedicreader.github.io/kosha/graph.html
0.015873 ['brave'] fts · GitHub Topics · GitHub https://github.com/topics/fts?o=desc&s=updated
0.016393 ['yahoo'] pypi.org › project › koshaskoshas · PyPI https://pypi.org/project/koshas/
0.015625 ['brave'] GitHub - colbymchenry/codegraph: Pre-indexed code knowledge graph, auto syncs on code changes, for Claude Code, Codex, Gemini, Cursor, OpenCode, AntiGravity, Kiro, and Hermes Agent — fewer tokens, fewer tool calls, 100% local https://github.com/colbymchenry/codegraph


In [ ]:
#| eval: false
res = research('sqlite WAL mode vs journal mode', n=5)
print(res['digest'][:500])
print([s['href'] for s in res['sources']])

## Write-Ahead Logging
https://www.sqlite.org/wal.html

Write-Ahead Logging

Table Of Contents

# 1\. Overview

The default method by which SQLite implements atomic commit and rollback is a rollback journal. Beginning with version 3.7.0 (2010-07-21), a new "Write-Ahead Log" option (hereafter referred to as "WAL") is available.

There are advantages and disadvantages to using WAL instead of a rollback journal. Advantages include:

1. WAL is significantly faster in most scenarios. 
  2. WAL provid
['https://www.sqlite.org/wal.html', 'https://blog.sqlite.ai/journal-modes-in-sqlite', 'https://mohit-bhalla.medium.com/understanding-wal-mode-in-sqlite-boosting-performance-in-sql-crud-operations-for-ios-5a8bd8be93d2', 'https://til.simonwillison.net/sqlite/enabling-wal-mode', 'https://fly.io/blog/wal-mode-in-litefs/']


In [ ]:
#| eval: false
page = read('https://en.wikipedia.org/wiki/Web_scraping')
print(page.text[:400])

The legality of web scraping varies across the world. In general, web scraping may be against the terms of service of some websites, but the enforceability of these terms is unclear.[11]

In the United States, website owners can use three major legal claims to prevent undesired web scraping: (1) copyright infringement (compilation), (2) violation of the Computer Fraud and Abuse Act ("CFAA"), and (


## One function for every source

Use `read` when you have a target and want its text. `read` first calls `what_is` to classify the target. The result is the name of a reader:

- `dir` for a local folder
- `file` for a local file
- `arxiv` for an arXiv URL or id
- `youtube` for a YouTube link
- `github` for a repository
- `ghfile` for a file on GitHub
- `pdf` for a URL that ends in `.pdf` or `/pdf`
- `web` for any other http or https URL

A path that exists on disk wins over every URL rule.

`read` returns the same seven fields for every supported target:

| field | holds |
|---|---|
| `ok` | True when the reader produced content |
| `kind` | the reader used, one of the names above |
| `title` | the title, or the first heading of the text |
| `source` | the URL, or the absolute path of a local file |
| `text` | the content as markdown |
| `skipped` | why nothing was read, when `ok` is False |
| `meta` | details from that reader, such as the authors of a paper or the HTTP status of a page, and the time it was read |

Here `read` opens this repository's own README:

In [ ]:
target = str(repo_root()/'README.md')
assert what_is(target) == 'file'
result = read(target)
assert sorted(result) == ['kind', 'meta', 'ok', 'skipped', 'source', 'text', 'title']
assert (result.ok, result.kind, result.title) == (True, 'file', 'fossick')
assert '# fossick' in result.text and result.skipped is None
result.kind, result.title, result.ok

A folder or GitHub repository contains multiple files. `read` does not combine them into one document. It returns `ok=True` with empty `text` and the local path in `meta['path']`. You choose which files to open. For a GitHub repository, that path is a fresh clone in `fossick`'s cache. With `pages=True`, a PDF returns a list of `(page_index, text)` pairs. Page indexes start at 0.

Sometimes a recognised target produces no content. A site can serve a bot check instead of the page. A YouTube video can have no English captions. A URL that looks like a PDF can return another file type. `read` reports these cases with `ok=False` and a reason in `skipped`. It does not raise. It raises `ValueError` only when the target is not one of the supported kinds.

## Modules

| notebook | what it covers |
|---|---|
| `00_core` | Download pages ([<code>fetch</code>](https://vedicreader.github.io/fossick/core.html#fetch) and [<code>crawl</code>](https://vedicreader.github.io/fossick/core.html#crawl)). Convert HTML to markdown with `to_md`. Read arXiv, GitHub, YouTube, and PDF sources. Find the JSON API behind a page. Use `read`. |
| `01_cdp` | Drive Chrome through the DevTools protocol. Capture page snapshots, fill forms, run steps, and capture network requests. |
| `02_search` | Search several engines. Merge and reorder results. Use `google` and [<code>research</code>](https://vedicreader.github.io/fossick/search.html#research). |
| `03_cli` | Use the `fossick` shell command. |
| `04_mcp` | Use the MCP server with coding agents. |
| `05_shop` | Add products to a shopping cart and fill checkout forms on a live store. |
| `06_quality` | Remove copycat sites, prefer authoritative sources, and split a long question into several searches. |

## Common calls

```python
search('query', n=10, region='auto', timelimit='y')   # results from the past year
research('query', n=5)                    # search, read the top pages, return a digest that cites them
fetch(url, auto=True)                     # try stronger download methods until one is not blocked
crawl(url, max_pages=5, same_domain=True) # follow links from url, staying on its site
google('query', n=10)                     # Google's own ranking, through the stealth browser
cdp = await cdp_connect(); pg = await cdp.open_page(url); print(await pg.snapshot())
```

From the shell: `fossick search "q"`, `fossick fetch URL --auto`, `fossick research "q"`. For agents: `uvx fossick-mcp`, or `fossick-mcp --http` to serve over HTTP instead of standard input and output.

## A logged-in Chrome

Some pages are shown only to logged-in users. `fossick` uses a separate copy of Chrome with remote debugging on port 9223. It uses the profile at `~/.cache/fastcdp/cdp-chrome`. `cdp_connect` starts Chrome the first time it is needed. Log in to a site once in that window. The profile keeps the cookies. Later calls such as `fetch(url, session=True)` use the logged-in session. `fetch(url, auto=True)` and `read` use this Chrome when every other method is blocked.

To start Chrome by hand in a visible window, run `await cdp_setup(headless=False)` and log in. The `fastcdp-setup` command comes with the `fastcdp` dependency. It adds a launcher for the same Chrome and profile to your applications.

## Untrusted URLs

`fetch(url, auto=True)` and `read` can use the logged-in Chrome on port 9223. `read` uses `auto=True` by default. This step starts Chrome when it is not running and sends its cookies for the site. Use this mode only for URLs that you chose. When a URL comes from a web page, document, or another person, pass `auto=False`:

```python
fetch(url, auto=False, max_bytes=20_000_000)   # plain HTTP only; stop after 20 MB
read(url, auto=False, max_bytes=20_000_000)    # also for PDFs, arXiv papers and GitHub files
```

With `auto=False`, `fossick` does not start a browser or connect to port 9223. This also applies to PDFs and redirects. The plain path refuses private and local addresses. It connects only to the checked address. A second DNS answer cannot send it to `127.0.0.1`. It checks each redirect in the same way. `max_bytes` and `max_recv_speed` limit download size and speed. Both limits are off by default. Through a proxy and in browser paths, `fossick` checks only the first URL.

On a machine that inspects TLS traffic with its own root certificate, `curl_cffi` does not trust that root. It does not use the system keychain. A download then fails with `curl: (60) ... unable to get local issuer certificate`. Set `SSL_CERT_FILE` to a CA bundle file that holds the normal roots and the inspection root. For example, run `export SSL_CERT_FILE=~/Library/Java/macos-all-ca.pem`. `fossick` also reads `CURL_CA_BUNDLE` and `REQUESTS_CA_BUNDLE`. You can pass `ca_bundle=` on each call. Verification stays on.
